In [1]:
#A*
import heapq
import math
from itertools import combinations

class TSP_AStar:
    def __init__(self, distance_matrix, start=0):
        """
        distance_matrix: ma trận khoảng cách giữa các thành phố
        start: thành phố bắt đầu (mặc định 0)
        """
        self.n = len(distance_matrix)
        self.dist = distance_matrix
        self.start = start

    def heuristic(self, visited, current):
        """
        Ước lượng chi phí từ trạng thái hiện tại đến khi kết thúc:
        - Nếu đã thăm hết thành phố: khoảng cách từ current về start
        - Ngược lại: chi phí cây khung nhỏ nhất của các thành phố chưa thăm + khoảng cách gần nhất từ current đến các thành phố chưa thăm
        """
        unvisited = [city for city in range(self.n) if city not in visited]

        if not unvisited:  # Đã thăm hết
            return self.dist[current][self.start]

        # Tính MST trên tập unvisited + current (dùng Prim)
        nodes = unvisited + [current]
        mst_cost = self.mst_prim(nodes)

        # Heuristic đơn giản: chi phí MST + khoảng cách gần nhất từ current đến unvisited
        min_to_unvisited = min(self.dist[current][city] for city in unvisited)

        return mst_cost + min_to_unvisited

    def mst_prim(self, nodes):
        """Tính chi phí cây khung nhỏ nhất trên tập nodes"""
        if len(nodes) <= 1:
            return 0

        n_nodes = len(nodes)
        node_to_idx = {node: i for i, node in enumerate(nodes)}

        # Ma trận khoảng cách con
        dist_sub = [[0]*n_nodes for _ in range(n_nodes)]
        for i in range(n_nodes):
            for j in range(n_nodes):
                dist_sub[i][j] = self.dist[nodes[i]][nodes[j]]

        # Prim's algorithm
        visited = [False]*n_nodes
        min_edge = [math.inf]*n_nodes
        min_edge[0] = 0
        total_cost = 0

        for _ in range(n_nodes):
            u = -1
            for i in range(n_nodes):
                if not visited[i] and (u == -1 or min_edge[i] < min_edge[u]):
                    u = i

            visited[u] = True
            total_cost += min_edge[u]

            for v in range(n_nodes):
                if not visited[v] and dist_sub[u][v] < min_edge[v]:
                    min_edge[v] = dist_sub[u][v]

        return total_cost

    def solve(self):
        """Giải TSP bằng A*"""

        start_state = (0, 0, (self.start,), self.start)
        pq = []
        heapq.heappush(pq, start_state)

        best_cost = math.inf
        best_path = None
        visited_states = {}

        while pq:
            f, g, visited, current = heapq.heappop(pq)


            state_key = (visited, current)
            if state_key in visited_states and visited_states[state_key] <= g:
                continue
            visited_states[state_key] = g


            if len(visited) == self.n:
                total = g + self.dist[current][self.start]
                if total < best_cost:
                    best_cost = total
                    best_path = list(visited) + [self.start]
                continue


            for nxt in range(self.n):
                if nxt in visited:
                    continue

                new_g = g + self.dist[current][nxt]
                if new_g >= best_cost:  # Cắt tỉa
                    continue

                new_visited = visited + (nxt,)
                h = self.heuristic(new_visited, nxt)
                new_f = new_g + h

                heapq.heappush(pq, (new_f, new_g, new_visited, nxt))

        return best_path, best_cost
if __name__ == "__main__":

    dist_matrix = [
        [0, 10, 15, 20],
        [10, 0, 35, 25],
        [15, 35, 0, 30],
        [20, 25, 30, 0]
    ]

    tsp_solver = TSP_AStar(dist_matrix, start=0)
    path, cost = tsp_solver.solve()

    print("Đường đi tối ưu tìm được:", path)
    print("Tổng chi phí:", cost)

Đường đi tối ưu tìm được: [0, 1, 3, 2, 0]
Tổng chi phí: 80
